# 🧪 W2-D1 概念实验：FFN、LayerNorm 与残差连接

> 配套阅读：`第2周-Day1-FFN-LayerNorm与残差连接.md`（完整原理、公式推导、业务关联在那边）
> 这个 notebook 用可执行实验回答三个问题：
> 1. **LayerNorm 到底做了什么？** 和 BatchNorm 有什么本质区别？
> 2. **残差连接为什么能救深层网络？** 没有它会怎样？
> 3. **FFN 的升维-降维有什么意义？** 每个位置独立处理是真的吗？

## 实验 1：LayerNorm vs BatchNorm — 归一化到底"沿哪个轴"？

- **LayerNorm**：对单个 token 的所有特征做归一化（沿特征维度），每个 token 独立
- **BatchNorm**：对 batch 内同一特征做归一化（沿 batch 维度），依赖 batch 统计量

这意味着 LayerNorm 在推理时（batch_size=1）行为完全一致，而 BatchNorm 会"失准"。

In [ ]:
import numpy as np

np.random.seed(42)

# 模拟 4 个 token、8 维特征
x = np.random.randn(4, 8) * 5 + np.array([10, -3, 0, 7, -8, 2, -1, 5])  # 不同 token 分布差异大
eps = 1e-5

# --- LayerNorm: 沿特征维度 (axis=-1) ---
def layer_norm(x, eps=1e-5):
    mean = x.mean(axis=-1, keepdims=True)
    var = x.var(axis=-1, keepdims=True)
    return (x - mean) / np.sqrt(var + eps)

# --- BatchNorm: 沿 batch 维度 (axis=0) ---
def batch_norm(x, eps=1e-5):
    mean = x.mean(axis=0, keepdims=True)   # 每个 feature 的 batch 均值
    var = x.var(axis=0, keepdims=True)
    return (x - mean) / np.sqrt(var + eps)

ln_out = layer_norm(x)
bn_out = batch_norm(x)

print("=== LayerNorm ===")
print("每个 token 的均值（应≈0）:", ln_out.mean(axis=-1).round(4))
print("每个 token 的标准差（应≈1）:", ln_out.std(axis=-1).round(4))
print()
print("=== BatchNorm ===")
print("每个 feature 的 batch 均值（应≈0）:", bn_out.mean(axis=0).round(4))
print("每个 feature 的 batch 标准差（应≈1）:", bn_out.std(axis=0).round(4))
print()

# 关键区别：推理时 batch_size=1，BatchNorm 用训练时的 running mean/var（这里模拟"完全不同分布"）
single_token = x[0:1]  # batch_size=1
print("=== 推理 batch_size=1 ===")
print("LayerNorm 仍然正常（均值0, 标准差1）:")
ln_single = layer_norm(single_token)
print(f"  均值={ln_single.mean():.4f}, 标准差={ln_single.std():.4f}")
print()
print("BatchNorm 在 batch=1 时退化为：")
bn_single = batch_norm(single_token)  # 单样本归一化：每个特征值减自己均值（即自己）→ 全0！
print(f"  结果全零？ {np.allclose(bn_single, 0)}")
print("  （实际工程中 BN 用 running statistics，但分布漂移时仍会失准）")

## 实验 2：残差连接 vs 无残差 — 梯度能否传过 50 层？

模拟一个 50 层的线性网络，每层是一个随机矩阵。对比有/无残差连接时，反向传播的梯度大小。

In [ ]:
import numpy as np

np.random.seed(0)
n_layers = 50
d = 64

# 生成随机权重（Xavier 初始化）
layers = [np.random.randn(d, d) * np.sqrt(2.0 / (d + d)) for _ in range(n_layers)]

# === 无残差连接：y = W50 · W49 · ... · W1 · x ===
x = np.random.randn(d)
y_no_res = x.copy()
for W in layers:
    y_no_res = W @ y_no_res

# === 有残差连接：y = x + W50 · (x + W49 · (x + ... W1 · x)) ===
y_res = x.copy()
for W in layers:
    y_res = y_res + W @ y_res

print(f"无残差：输出范数 = {np.linalg.norm(y_no_res):.6f}")
print(f"有残差：输出范数 = {np.linalg.norm(y_res):.6f}")
print(f"输入范数 = {np.linalg.norm(x):.6f}")
print()

# 梯度分析：
# 无残差：梯度 ∝ ∏ Wᵀ，范数指数衰减或爆炸
# 有残差：梯度至少有一条路径是恒等映射（系数=1），不会被压到 0
# 用数值验证：给输出一个小的扰动，看输入的等效梯度大小

delta = 1.0  # 输出扰动的范数

# 无残差：dL/dx = W1ᵀ · W2ᵀ · ... · W50ᵀ · dL/dy
grad_no_res = np.ones(d)
for W in reversed(layers):
    grad_no_res = W.T @ grad_no_res

# 有残差：梯度 = ∏(I + Wᵀ)，每层至少有 I 的贡献
grad_res = np.ones(d)
for W in reversed(layers):
    grad_res = grad_res + W.T @ grad_res

print(f"无残差：梯度范数 = {np.linalg.norm(grad_no_res):.2e}")
print(f"有残差：梯度范数 = {np.linalg.norm(grad_res):.2e}")
print()
ratio_no = np.linalg.norm(grad_no_res) / np.linalg.norm(np.ones(d))
ratio_res = np.linalg.norm(grad_res) / np.linalg.norm(np.ones(d))
print(f"无残差梯度衰减到初始的 {ratio_no:.2e}")
print(f"有残差梯度相对于初始的 {ratio_res:.2e}")
print("\n结论：残差连接通过恒等映射的捷径，让梯度不会在深层网络中消失。")

## 实验 3：FFN 的升维-降维 — 每个位置独立处理

FFN 对序列中每个 token 独立做相同的线性变换。验证：
1. 输入不同位置的 token，FFN 输出互不影响
2. 参数量 = 2 × d_model × d_ff（W1 + W2）

In [ ]:
import numpy as np

np.random.seed(7)
d_model = 64
d_ff = 256  # 4 × d_model

W1 = np.random.randn(d_model, d_ff) * np.sqrt(2.0 / d_model)
b1 = np.zeros(d_ff)
W2 = np.random.randn(d_ff, d_model) * np.sqrt(2.0 / d_ff)
b2 = np.zeros(d_model)

def gelu(x):
    return 0.5 * x * (1 + np.tanh(np.sqrt(2 / np.pi) * (x + 0.044715 * x**3)))

def ffn(x):
    h = gelu(x @ W1 + b1)
    return h @ W2 + b2

# 3 个 token 的输入
tokens = np.random.randn(3, d_model)
out_all = ffn(tokens)

# 单独处理第 1 个 token
out_0_alone = ffn(tokens[0:1])[0]
out_1_alone = ffn(tokens[1:2])[0]

print("逐 token 处理 vs 批量处理结果一致？")
print(f"  token 0: {np.allclose(out_all[0], out_0_alone)}")
print(f"  token 1: {np.allclose(out_all[1], out_1_alone)}")
print()

# 参数量计算
params_ffn = 2 * d_model * d_ff + d_ff + d_model
print(f"d_model={d_model}, d_ff={d_ff}")
print(f"FFN 参数量: W1({d_model}×{d_ff}) + W2({d_ff}×{d_model}) + biases = {params_ffn:,}")
print(f"其中 W1+W2 占 {2*d_model*d_ff:,} ({2*d_model*d_ff/params_ffn*100:.1f}%)")
print()

# GPT-2 Small 级别的估算
d_m, d_f, n_layers = 768, 3072, 12
attn_params = n_layers * (4 * d_m * d_m)  # Q,K,V,O projections
ffn_params_total = n_layers * (2 * d_m * d_f)
embed_params = 50257 * d_m  # vocab × d_model
total = attn_params + ffn_params_total + embed_params
print(f"GPT-2 Small 估算 (d={d_m}, layers={n_layers}):")
print(f"  FFN 总参数: {ffn_params_total/1e6:.1f}M ({ffn_params_total/total*100:.0f}%)")
print(f"  Attention 总参数: {attn_params/1e6:.1f}M ({attn_params/total*100:.0f}%)")
print(f"  Embedding 参数: {embed_params/1e6:.1f}M ({embed_params/total*100:.0f}%)")
print(f"  总计: {total/1e6:.1f}M")

## 实验 4：LayerNorm + 残差组合 — Pre-Norm vs Post-Norm

- Post-Norm（BERT）: `x = LN(x + Sublayer(x))`
- Pre-Norm（GPT-2/LLaMA）: `x = x + Sublayer(LN(x))`

用数值模拟看两种方式下输出数值的稳定性。

In [ ]:
import numpy as np

np.random.seed(42)

def layer_norm(x, eps=1e-5):
    m = x.mean(axis=-1, keepdims=True)
    v = x.var(axis=-1, keepdims=True)
    return (x - m) / np.sqrt(v + eps)

d = 64
n_layers = 20

# 简单的随机线性子层（模拟 Attention 或 FFN）
def sublayer(x):
    W = np.random.randn(d, d) * 0.05
    return x @ W

x0 = np.random.randn(1, d)

# Post-Norm
x_post = x0.copy()
post_norms = []
for _ in range(n_layers):
    x_post = layer_norm(x_post + sublayer(x_post))
    post_norms.append(np.linalg.norm(x_post))

# Pre-Norm
x_pre = x0.copy()
pre_norms = []
for _ in range(n_layers):
    x_pre = x_pre + sublayer(layer_norm(x_pre))
    pre_norms.append(np.linalg.norm(x_pre))

print("各层输出范数（观察稳定性）:")
print(f"{'层':>3}  {'Post-Norm':>10}  {'Pre-Norm':>10}")
for i in range(n_layers):
    print(f"{i+1:3d}  {post_norms[i]:10.4f}  {pre_norms[i]:10.4f}")
print()
print(f"Post-Norm 最终范数: {post_norms[-1]:.4f}")
print(f"Pre-Norm 最终范数: {pre_norms[-1]:.4f}")
print("\n两者都保持稳定，但 Pre-Norm 不对残差通路做归一化，梯度流更顺畅。")
print("这就是现代大模型（LLaMA、Qwen）都选 Pre-Norm 的原因。")